# Figure 1. Posterior alpha power during eyes-open and eyes-closed rest

Run all cells in Python 3 with NumPy, pandas, SciPy, and Matplotlib installed. The run-level values from `corrected_alpha_results.csv` are embedded below so this notebook runs independently of the raw EEG files. The analysis averages two runs per condition for each participant. It assumes the stored integrated powers are in V² and converts them to µV². The next cell reproduces the draft's descriptive statistics and paired test. The final cell saves a 300 dpi PNG in the current working directory.


In [ ]:
from io import StringIO

import pandas as pd

run_results_csv = """
participant,condition,run,alpha_power
sub-05,ses-OA,1,4.261111218526347e-12
sub-05,ses-OA,2,6.9117584320968695e-12
sub-05,ses-OC,1,7.608407215422375e-12
sub-05,ses-OC,2,4.841588335134324e-12
sub-06,ses-OA,1,3.3724767375001465e-11
sub-06,ses-OA,2,3.823962173265172e-11
sub-06,ses-OC,1,9.611355415778817e-11
sub-06,ses-OC,2,5.428475254771638e-11
sub-07,ses-OA,1,1.0954408317138347e-11
sub-07,ses-OA,2,1.184668611921782e-11
sub-07,ses-OC,1,1.4445916259793934e-11
sub-07,ses-OC,2,9.726211043941956e-12
sub-08,ses-OA,1,4.167445598196446e-12
sub-08,ses-OA,2,5.658174885205657e-12
sub-08,ses-OC,1,1.888300191697908e-11
sub-08,ses-OC,2,8.634712841011948e-12
sub-09,ses-OA,1,7.75620614933841e-11
sub-09,ses-OA,2,7.030244311799982e-11
sub-09,ses-OC,1,7.83468872801812e-11
sub-09,ses-OC,2,7.566485392189357e-11
sub-10,ses-OA,1,2.3812063163826162e-11
sub-10,ses-OA,2,2.3616855321660137e-11
sub-10,ses-OC,1,5.0176643309367917e-11
sub-10,ses-OC,2,4.809577820066783e-11
sub-11,ses-OA,1,5.982154722561407e-11
sub-11,ses-OA,2,7.00861311650365e-11
sub-11,ses-OC,1,9.408440733059221e-11
sub-11,ses-OC,2,1.0098179579734605e-10
sub-12,ses-OA,1,1.0646903675242177e-11
sub-12,ses-OA,2,1.587898428011805e-11
sub-12,ses-OC,1,3.4895936703096945e-11
sub-12,ses-OC,2,3.449780744797761e-11
sub-13,ses-OA,1,3.6514525869445483e-11
sub-13,ses-OA,2,8.044283124920391e-11
sub-13,ses-OC,1,6.474233889074636e-11
sub-13,ses-OC,2,1.2826991320807393e-10
sub-14,ses-OA,1,1.4098762411610313e-11
sub-14,ses-OA,2,1.3378125587821422e-11
sub-14,ses-OC,1,5.318130601276332e-11
sub-14,ses-OC,2,5.42547443161428e-11
sub-15,ses-OA,1,3.0515519966206163e-11
sub-15,ses-OA,2,4.335811522585603e-11
sub-15,ses-OC,1,4.610316215401877e-11
sub-15,ses-OC,2,4.604562969151045e-11
sub-16,ses-OA,1,4.601459988347759e-11
sub-16,ses-OA,2,3.7177352380888145e-11
sub-16,ses-OC,1,3.251571712177509e-11
sub-16,ses-OC,2,4.960181827772423e-11
sub-17,ses-OA,1,6.711217563971743e-11
sub-17,ses-OA,2,6.792660982083733e-11
sub-17,ses-OC,1,1.6472325699984693e-10
sub-17,ses-OC,2,6.179616112828748e-11
sub-18,ses-OA,1,1.3247548105210428e-11
sub-18,ses-OA,2,1.893696595145793e-11
sub-18,ses-OC,1,1.3713330604770506e-11
sub-18,ses-OC,2,1.367793491125622e-11
sub-19,ses-OA,1,1.4308197643836807e-11
sub-19,ses-OA,2,1.5012187531447033e-11
sub-19,ses-OC,1,5.0136401618330075e-11
sub-19,ses-OC,2,5.63167801397278e-11
sub-20,ses-OA,1,3.548644467081306e-12
sub-20,ses-OA,2,4.125524382953783e-12
sub-20,ses-OC,1,1.2741728920280539e-11
sub-20,ses-OC,2,5.685752635413045e-12
sub-21,ses-OA,1,1.112269070539943e-11
sub-21,ses-OA,2,1.349629850067047e-11
sub-21,ses-OC,1,1.8153660677106335e-11
sub-21,ses-OC,2,2.3115014840639886e-11
sub-22,ses-OA,1,7.60726266696446e-12
sub-22,ses-OA,2,9.771107637819693e-12
sub-22,ses-OC,1,2.3080442727016374e-11
sub-22,ses-OC,2,1.5509406984229592e-11
sub-24,ses-OA,1,3.548607084880456e-11
sub-24,ses-OA,2,3.6753054747889936e-11
sub-24,ses-OC,1,3.449050186653247e-11
sub-24,ses-OC,2,3.76792188049561e-11
sub-25,ses-OA,1,8.124604766455088e-12
sub-25,ses-OA,2,1.4740915581846327e-11
sub-25,ses-OC,1,3.081240744693539e-11
sub-25,ses-OC,2,1.9519505737079256e-11
sub-26,ses-OA,1,2.073070439639887e-11
sub-26,ses-OA,2,2.270012505310837e-11
sub-26,ses-OC,1,6.231864178289491e-11
sub-26,ses-OC,2,8.984611864733728e-11
sub-27,ses-OA,1,4.072828139722192e-11
sub-27,ses-OA,2,4.6514045223970756e-11
sub-27,ses-OC,1,1.1133360775167642e-10
sub-27,ses-OC,2,1.1779194428529444e-10
sub-28,ses-OA,1,4.829413834684969e-12
sub-28,ses-OA,2,6.228349960135767e-12
sub-28,ses-OC,1,1.4205990968392052e-11
sub-28,ses-OC,2,5.063074370745437e-12
sub-29,ses-OA,1,3.611491313730906e-11
sub-29,ses-OA,2,5.053955608907415e-11
sub-29,ses-OC,1,7.908391521229982e-11
sub-29,ses-OC,2,2.0433864177087686e-10
sub-30,ses-OA,1,8.81205343591842e-12
sub-30,ses-OA,2,1.0807967125322298e-11
sub-30,ses-OC,1,9.293474635562337e-11
sub-30,ses-OC,2,2.7015042089856748e-11
sub-31,ses-OA,1,1.1991442367267845e-12
sub-31,ses-OA,2,1.160640841989343e-12
sub-31,ses-OC,1,1.570833593670097e-12
sub-31,ses-OC,2,1.8449757481572803e-12
sub-33,ses-OA,1,1.1948149243092476e-11
sub-33,ses-OA,2,1.4142893362894095e-11
sub-33,ses-OC,1,2.325615734897862e-11
sub-33,ses-OC,2,2.6244711012031985e-11
sub-34,ses-OA,1,1.3329573758762559e-11
sub-34,ses-OA,2,2.1677490851844594e-11
sub-34,ses-OC,1,5.2243428127770104e-11
sub-34,ses-OC,2,3.4556623197694373e-11
sub-35,ses-OA,1,6.485538026281996e-12
sub-35,ses-OA,2,6.183936157368307e-12
sub-35,ses-OC,1,1.3715638270526677e-11
sub-35,ses-OC,2,1.3014200228714441e-11
sub-36,ses-OA,1,9.28333039126008e-12
sub-36,ses-OA,2,1.1579208867303398e-11
sub-36,ses-OC,1,2.480872402510139e-11
sub-36,ses-OC,2,1.6892025688036806e-11
sub-37,ses-OA,1,3.509840515621752e-12
sub-37,ses-OA,2,2.95723683281983e-12
sub-37,ses-OC,1,6.1938553492018484e-12
sub-37,ses-OC,2,4.17496822689081e-12
sub-38,ses-OA,1,7.476247193016418e-12
sub-38,ses-OA,2,8.200575807679948e-12
sub-38,ses-OC,1,8.32675351229995e-12
sub-38,ses-OC,2,7.191439764220427e-12
sub-39,ses-OA,1,5.7622895029983e-12
sub-39,ses-OA,2,5.346234430748059e-12
sub-39,ses-OC,1,5.410474863240691e-12
sub-39,ses-OC,2,7.107128674408427e-12
sub-40,ses-OA,1,1.1770702113445947e-11
sub-40,ses-OA,2,1.8403555389662355e-11
sub-40,ses-OC,1,2.973400373567024e-11
sub-40,ses-OC,2,2.686498582929486e-11
sub-41,ses-OA,1,1.6287256232992653e-11
sub-41,ses-OA,2,1.9333082207980953e-11
sub-41,ses-OC,1,4.118666612218871e-11
sub-41,ses-OC,2,3.408627173120921e-11
sub-42,ses-OA,1,9.287278873603355e-12
sub-42,ses-OA,2,1.658785643505049e-11
sub-42,ses-OC,1,3.520324663338346e-11
sub-42,ses-OC,2,1.9454171553221275e-11
sub-43,ses-OA,1,4.980463712731898e-12
sub-43,ses-OA,2,4.616599264113551e-12
sub-43,ses-OC,1,1.1282359341260837e-11
sub-43,ses-OC,2,9.32990095695475e-12
sub-44,ses-OA,1,7.809727098284183e-12
sub-44,ses-OA,2,5.961966912327549e-12
sub-44,ses-OC,1,5.0474208642193636e-12
sub-44,ses-OC,2,5.618342855882186e-12
sub-45,ses-OA,1,6.155098793241288e-11
sub-45,ses-OA,2,6.416405611919937e-11
sub-45,ses-OC,1,7.971135473641974e-11
sub-45,ses-OC,2,9.293291866201017e-11
sub-46,ses-OA,1,3.2037405495603254e-12
sub-46,ses-OA,2,4.8916874583541545e-12
sub-46,ses-OC,1,1.2174923289147405e-11
sub-46,ses-OC,2,1.0378246643559919e-11
sub-47,ses-OA,1,2.75539943053491e-11
sub-47,ses-OA,2,4.287902051313643e-11
sub-47,ses-OC,1,5.5198175216634086e-11
sub-47,ses-OC,2,4.3286024016785e-11
sub-48,ses-OA,1,1.1437849184937211e-11
sub-48,ses-OA,2,1.0144101558493525e-11
sub-48,ses-OC,1,1.149891111783806e-11
sub-48,ses-OC,2,1.3442295724638636e-11
sub-49,ses-OA,1,8.665133356430256e-12
sub-49,ses-OA,2,1.2263464276463711e-11
sub-49,ses-OC,1,1.4097085557435426e-11
sub-49,ses-OC,2,9.406753978914942e-12
"""
runs = pd.read_csv(StringIO(run_results_csv))


In [ ]:
import numpy as np
from scipy.stats import ttest_rel

condition_order = ["ses-OA", "ses-OC"]
conditions = ["Eyes open", "Eyes closed"]
run_counts = runs.groupby(["participant", "condition"])["run"].agg(list)
assert len(run_counts) == 86 and all(sorted(runs) == [1, 2] for runs in run_counts)
paired = (
    runs.groupby(["participant", "condition"])["alpha_power"]
    .mean()
    .unstack()
    .loc[:, condition_order]
    .mul(1e12)
)
assert paired.shape == (43, 2) and (paired > 0).all().all()
paired.columns = conditions

test = ttest_rel(np.log(paired["Eyes closed"]), np.log(paired["Eyes open"]))
increased = (paired["Eyes closed"] > paired["Eyes open"]).sum()

print(paired.mean().round(2).to_string())
print(f"Higher with eyes closed: {increased}/{len(paired)}")
print(f"Paired t test on log power: t({test.df:.0f}) = {test.statistic:.2f}, p = {test.pvalue:.3g}")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(5.5, 4.8), layout="constrained")
x = [0, 1]

for eyes_open, eyes_closed in paired.itertuples(index=False, name=None):
    color = "#21758A" if eyes_closed > eyes_open else "#AC5630"
    ax.plot(x, [eyes_open, eyes_closed], color=color, alpha=0.52,
            linewidth=1, marker="o", markersize=3)

ax.scatter(x, paired.mean(), marker="D", s=65, color="#20252A",
           edgecolor="white", linewidth=0.8, zorder=4)
ax.set(
    xticks=x, xticklabels=conditions, xlim=(-0.25, 1.25), ylim=(0, 150),
    ylabel="Absolute posterior alpha power (µV²)",
)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E4E7EA")
ax.set_axisbelow(True)
ax.legend(handles=[
    Line2D([], [], color="#21758A", label="Higher with eyes closed"),
    Line2D([], [], color="#AC5630", label="Lower with eyes closed"),
    Line2D([], [], color="#20252A", marker="D", linestyle="none", label="Condition mean"),
], loc="upper left", frameon=False, fontsize=9)
fig.savefig("Figure_1_Paired_Posterior_Alpha.png", dpi=300)
plt.show()
